# Bloco 4 — Otimização de Escalas

Demonstração do pipeline de escalonamento semanal de equipas de Housekeeping.

- Carrega a semana-alvo e os parâmetros operacionais
- Prevê necessidades de cobertura por slot (Gradient Boosting)
- Resolve o modelo ILP híbrido (internos + externos se necessário)
- Apresenta a escala gerada e os KPIs operacionais

> A lógica do modelo está disponível em `src/optimizer.py`.

## 1. Configuração

In [40]:
from pathlib import Path
import sys, joblib, pandas as pd

ROOT_DIR    = Path.cwd().parent
DATA_DIR    = ROOT_DIR / "data" / "synthetic"
OUTPUTS_DIR = ROOT_DIR / "outputs"
SRC_DIR     = ROOT_DIR / "src"

# Verificar dependências do Bloco 3
em_falta = [
    f.name for f in [
        OUTPUTS_DIR / "gb_config.pkl",
        OUTPUTS_DIR / "gb_model.pkl",
        OUTPUTS_DIR / "gb_model_v2.pkl",
        OUTPUTS_DIR / "gb_features.pkl",
    ] if not f.exists()
]
if em_falta:
    raise FileNotFoundError(f"Executar primeiro o Bloco 3. Em falta: {em_falta}")

sys.path.insert(0, str(SRC_DIR))
from optimizer import (
    preparar_parametros, prever_cobertura,
    construir_modelo, resolver,
    extrair_escala, calcular_kpis,
    mostrar_escala, imprimir_kpis,
)

# w1 = preferências | w2 = cobertura ideal | w3 = equidade de carga
w1, w2, w3 = 5, 10, 2

print("✅ Pronto —", ROOT_DIR)

✅ Pronto — c:\Projetos\Projeto Final\hotel_scheduling


## 2. Semana-alvo

Carregada automaticamente a partir do artefacto exportado no Bloco 3.

In [41]:
config = joblib.load(OUTPUTS_DIR / "gb_config.pkl")

SEMANA_INICIO = config["semana_inicio"]
SEMANA_FIM    = config["semana_fim"]
datas         = pd.date_range(SEMANA_INICIO, SEMANA_FIM)
D             = [d.strftime("%Y-%m-%d") for d in datas]

print(f"Semana : {SEMANA_INICIO} → {SEMANA_FIM}")

Semana : 2025-01-01 → 2025-01-07


## 3. Parâmetros operacionais

In [42]:
params = preparar_parametros(DATA_DIR, datas)

Parâmetros preparados:
  20 colaboradores | 7 dias | Turnos: ['T1', 'T2']
  3 funções | 5 recursos externos
  Custo fixo semanal: 6781.62€  [comprometido]


## 4. Previsão de cobertura

O modelo GB prevê `N_min` (hard constraint H1) e `N_ideal` (soft constraint S4) para cada um dos 42 slots semanais.

In [43]:
N_min, N_ideal, df_prev = prever_cobertura(DATA_DIR, datas, params, OUTPUTS_DIR)
df_prev.head(10)

Previsão de cobertura:
  42 slots previstos | 38/42 previsões exactas
  N_min total: 77 turnos


,data,turno_id,funcao,N_min_real,N_min_previsto,diff
0,2025-01-01,T1,Auxiliar de limpeza,1,1,0
1,2025-01-01,T1,Empregada de andares,4,4,0
2,2025-01-01,T1,Supervisora,2,2,0
3,2025-01-01,T2,Auxiliar de limpeza,1,1,0
4,2025-01-01,T2,Empregada de andares,2,2,0
5,2025-01-01,T2,Supervisora,1,1,0
6,2025-01-02,T1,Auxiliar de limpeza,1,1,0
7,2025-01-02,T1,Empregada de andares,4,3,-1
8,2025-01-02,T1,Supervisora,2,2,0
9,2025-01-02,T2,Auxiliar de limpeza,1,1,0


## 5. Construção e resolução do modelo

Modelo híbrido: internos prioritários, externos ativados apenas quando necessário para satisfazer H1.

In [44]:
prob, x, y, delta, e = construir_modelo(params, N_min, N_ideal, D, w1, w2, w3)
status, tempo = resolver(prob)

Modelo construído:
  Variáveis internas (x): 466
  Variáveis externas (y): 118
  Restrições: 516
  Pesos: w1=5  w2=10  w3=2
Solver: Optimal  |  Tempo: 0.15s


## 6. Escala semanal

In [45]:
escala, escala_int, escala_ext = extrair_escala(x, y)
mostrar_escala(escala)

Escala extraída: 90 alocações internas | 0 externas (0 recursos)


data           2025-01-01      2025-01-02      2025-01-03      2025-01-04  \
turno_id               T1   T2         T1   T2         T1   T2         T1   
colaborador_id                                                              
C001                    —  Sup          —    —          —  Sup          —   
C002                    —  Emp          —    —          —  Emp          —   
C003                    —    —        Emp    —          —    —        Emp   
C004                  Emp    —          —    —        Emp    —          —   
C005                  Aux    —          —    —        Aux    —        Aux   
C006                    —    —        Emp    —        Emp    —        Emp   
C008                  Emp    —        Emp    —          —    —          —   
C009                    —  Emp          —  Sup          —  Sup          —   
C010                  Emp    —          —    —        Emp    —        Emp   
C011                    —    —        Sup    —        Sup    —        Emp   
C012                    —  Sup          —  Emp          —  Emp          —   
C013                    —    —          —  Emp          —  Emp          —   
C014                  Aux    —        Aux    —          —    —        Aux   
C015                    —  Emp          —  Aux          —    —          —   
C016                  Sup    —          —    —        Sup    —          —   
C017                  Sup    —        Sup    —          —    —        Sup   
C018                    —  Aux          —    —          —  Aux          —   
C020                  Emp    —          —    —        Emp    —        Sup   

data                2025-01-05      2025-01-06      2025-01-07       
turno_id         T2         T1   T2         T1   T2         T1   T2  
colaborador_id                                                       
C001              —          —  Emp          —  Emp          —  Emp  
C002            Sup          —  Emp          —    —          —  Emp  
C003              —        Emp    —        Emp    —        Emp    —  
C004              —          —  Aux        Emp    —        Emp    —  
C005              —          —    —        Aux    —        Aux    —  
C006              —        Emp    —          —    —        Emp    —  
C008              —        Emp    —        Emp    —        Emp    —  
C009            Sup          —    —          —    —          —  Sup  
C010              —        Sup    —        Emp    —          —    —  
C011              —          —    —        Sup    —        Sup    —  
C012              —          —  Sup          —  Emp          —    —  
C013            Emp          —  Emp          —    —          —  Emp  
C014              —        Aux    —          —    —        Aux    —  
C015            Emp          —  Aux          —    —          —  Aux  
C016            Aux        Emp    —        Sup    —          —    —  
C017              —        Sup    —          —    —        Sup    —  
C018            Emp          —    —          —  Aux          —  Sup  
C020              —        Emp    —          —  Sup          —    —

## 7. KPIs operacionais

In [46]:
kpis = calcular_kpis(
    escala, escala_ext, params,
    N_min, N_ideal,
    D, params["T"], params["F"],
    status,
)
imprimir_kpis(kpis, SEMANA_INICIO, SEMANA_FIM)

  KPIs — Escala semanal
  Semana          : 2025-01-01 a 2025-01-07
  Status          : Optimal
-------------------------------------------------------
  Custo fixo      : 6781.62 €  [comprometido]
  Custo externos  : 0.00 €  [incremental]
  Custo total     : 6781.62 €
-------------------------------------------------------
  Cobertura mín.  : 42/42 (100%)
  Violações H1    : 0
  Défice ideal    : 21
-------------------------------------------------------
  Internos escal. : 18
  Externos activ. : 0
  Total alocações : 90
